<a href="https://colab.research.google.com/github/michelleasilveira/ST554-Project2/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project 2 — Big Data Management
Michelle Aparecida Silveira

## Introduction

This project explores two core concepts in big data processing using **PySpark**:

**Part I** focuses on building a reusable data quality tool from scratch. Rather than writing one-off Spark scripts, we design a Python class called `SparkDataCheck` that wraps a Spark SQL DataFrame and exposes clean methods for validating and summarizing data. We then put it to work on a real-world air quality dataset.

**Part II** shifts to analysis. Using NFL quarterback statistics from 2005 to 2023, we compute season-level aggregates and ranking metrics — first with the **pandas-on-Spark** API, then again with the native **Spark SQL** API — so we can compare how the two interfaces handle the same task.

---
## Part I — Creating the `SparkDataCheck` Class

The goal here is to build a class that a data analyst could drop into any PySpark project to quickly check data quality without writing boilerplate Spark code every time.

The class supports two ways to load data (from a CSV file or from an existing pandas DataFrame), three **validation methods** that append boolean flag columns to the DataFrame, and two **summarization methods** that return results as standard pandas DataFrames for easy inspection.

Below we write the class to a `.py` file so it can be imported as a module.

In [1]:
%%writefile SparkDataCheck.py
# Creating Python code from colab
"""
SparkDataCheck.py
-----------------
A data quality class that wraps a Spark SQL DataFrame and provides
methods for validating and summarizing data.

Author: Michelle A Silveira
"""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd


class SparkDataCheck:
    """
    A class that wraps a Spark SQL DataFrame and provides functionality
    for cleaning and checking data quality.

    Attributes
    ----------
    df : pyspark.sql.DataFrame
        The underlying Spark SQL DataFrame.
    """

    # Column types recognized as numeric
    _NUMERIC_TYPES = {
        "float", "int", "long", "bigint", "double", "integer",
        "smallint", "tinyint", "short", "longint"
    }

    def __init__(self, dataframe):
        """
        Initialize SparkDataCheck with a Spark DataFrame.

        Parameters
        ----------
        dataframe : pyspark.sql.DataFrame
            The Spark SQL DataFrame to wrap.
        """
        self.df = dataframe

    @classmethod
    def from_csv(cls, spark, path):
        """
        Create a SparkDataCheck instance by reading a CSV file.

        Parameters
        ----------
        spark : SparkSession
            The active Spark session.
        path : str
            Path to the CSV file (local or HDFS).

        Returns
        -------
        SparkDataCheck
            A new SparkDataCheck instance wrapping the loaded DataFrame.
        """
        df = spark.read.load(
            path,
            format="csv",
            header=True,
            inferSchema=True
        )
        return cls(df)

    @classmethod
    def from_pandas(cls, spark, pandas_df):
        """
        Create a SparkDataCheck instance from a standard pandas DataFrame.

        Parameters
        ----------
        spark : SparkSession
            The active Spark session.
        pandas_df : pandas.DataFrame
            A standard (non-Spark) pandas DataFrame.

        Returns
        -------
        SparkDataCheck
            A new SparkDataCheck instance wrapping the converted DataFrame.
        """
        df = spark.createDataFrame(pandas_df)
        return cls(df)

    # ------------------------------------------------------------------ #
    #  Private helpers                                                     #
    # ------------------------------------------------------------------ #

    def _is_numeric(self, col_name):
        """Return True if the column is a numeric type."""
        col_types = dict(self.df.dtypes)
        if col_name not in col_types:
            return False
        dtype = col_types[col_name].lower()
        # Handle decimal(precision, scale)
        if dtype.startswith("decimal"):
            return True
        return dtype in self._NUMERIC_TYPES

    def _is_string(self, col_name):
        """Return True if the column is a string type."""
        col_types = dict(self.df.dtypes)
        if col_name not in col_types:
            return False
        return col_types[col_name].lower() == "string"

    # ------------------------------------------------------------------ #
    #  Validation methods (return self for chaining)                      #
    # ------------------------------------------------------------------ #

    def check_numeric_bounds(self, col_name, lower=None, upper=None):
        """
        Check whether each value in a numeric column falls within
        user-defined bounds (inclusive).  Appends a boolean column named
        ``<col_name>_in_bounds`` to the DataFrame.  NULL inputs produce
        NULL outputs.

        Parameters
        ----------
        col_name : str
            The numeric column to validate.
        lower : numeric, optional
            Inclusive lower bound.  At least one of lower/upper must be given.
        upper : numeric, optional
            Inclusive upper bound.  At least one of lower/upper must be given.

        Returns
        -------
        self : SparkDataCheck
            Returns itself (chainable).
        """
        if lower is None and upper is None:
            print("Error: At least one of 'lower' or 'upper' must be provided.")
            return self

        if not self._is_numeric(col_name):
            print(
                f"Column '{col_name}' is not numeric "
                f"(type: {dict(self.df.dtypes).get(col_name, 'unknown')}). "
                "No modification made."
            )
            return self

        result_col = f"{col_name}_in_bounds"
        col = F.col(col_name)

        if lower is not None and upper is not None:
            check_expr = F.when(col.isNull(), None).otherwise(col.between(lower, upper))
        elif lower is not None:
            check_expr = F.when(col.isNull(), None).otherwise(col >= lower)
        else:
            check_expr = F.when(col.isNull(), None).otherwise(col <= upper)

        self.df = self.df.withColumn(result_col, check_expr)
        return self

    def check_string_levels(self, col_name, levels):
        """
        Check whether each value in a string column belongs to a set of
        valid levels.  Appends a boolean column named
        ``<col_name>_valid_level`` to the DataFrame.  NULL inputs produce
        NULL outputs.

        Parameters
        ----------
        col_name : str
            The string column to validate.
        levels : list of str
            The allowed values for the column.

        Returns
        -------
        self : SparkDataCheck
            Returns itself (chainable).
        """
        if not self._is_string(col_name):
            print(
                f"Column '{col_name}' is not a string column "
                f"(type: {dict(self.df.dtypes).get(col_name, 'unknown')}). "
                "No modification made."
            )
            return self

        result_col = f"{col_name}_valid_level"
        col = F.col(col_name)

        check_expr = F.when(col.isNull(), None).otherwise(col.isin(levels))
        self.df = self.df.withColumn(result_col, check_expr)
        return self

    def check_missing(self, col_name):
        """
        Check whether each value in a column is NULL.  Appends a boolean
        column named ``<col_name>_is_missing`` to the DataFrame.

        Parameters
        ----------
        col_name : str
            The column to check for missing values.

        Returns
        -------
        self : SparkDataCheck
            Returns itself (chainable).
        """
        result_col = f"{col_name}_is_missing"
        self.df = self.df.withColumn(result_col, F.col(col_name).isNull())
        return self

    # ------------------------------------------------------------------ #
    #  Summarization methods (return a pandas DataFrame)                  #
    # ------------------------------------------------------------------ #

    def numeric_summary(self, col_name=None, group_by=None):
        """
        Report the min and max of a numeric column (or all numeric columns).
        Returns a standard pandas DataFrame.

        Parameters
        ----------
        col_name : str, optional
            The numeric column to summarize.  If None, all numeric columns
            in the DataFrame are summarized.
        group_by : str, optional
            An optional column to group results by.

        Returns
        -------
        pandas.DataFrame or None
            A pandas DataFrame containing min/max values, or None if the
            specified column is not numeric.
        """
        if col_name is not None:
            # --- single column ---
            if not self._is_numeric(col_name):
                print(f"Column '{col_name}' is not numeric.")
                return None

            agg_exprs = [
                F.min(col_name).alias(f"{col_name}_min"),
                F.max(col_name).alias(f"{col_name}_max"),
            ]
            if group_by is not None:
                result = self.df.groupBy(group_by).agg(*agg_exprs).orderBy(group_by)
            else:
                result = self.df.agg(*agg_exprs)

            return result.toPandas()

        else:
            # --- all numeric columns ---
            numeric_cols = [
                c for c, t in self.df.dtypes
                if t.lower() in self._NUMERIC_TYPES or t.lower().startswith("decimal")
            ]

            if not numeric_cols:
                return pd.DataFrame()

            if group_by is not None:
                # Compute per-column and merge on group_by
                partials = []
                for c in numeric_cols:
                    agg_exprs = [
                        F.min(c).alias(f"{c}_min"),
                        F.max(c).alias(f"{c}_max"),
                    ]
                    part = (
                        self.df.groupBy(group_by)
                        .agg(*agg_exprs)
                        .orderBy(group_by)
                        .toPandas()
                    )
                    partials.append(part)
                combined = reduce(
                    lambda left, right: pd.merge(left, right, on=group_by), partials
                )
            else:
                agg_exprs = []
                for c in numeric_cols:
                    agg_exprs.append(F.min(c).alias(f"{c}_min"))
                    agg_exprs.append(F.max(c).alias(f"{c}_max"))
                combined = self.df.agg(*agg_exprs).toPandas()

            return combined

    def string_counts(self, col1, col2=None):
        """
        Report value counts for one or two string columns.
        Returns a standard pandas DataFrame.

        Parameters
        ----------
        col1 : str
            The required string column.
        col2 : str, optional
            An optional second string column for cross-tabulation.

        Returns
        -------
        pandas.DataFrame or None
            A pandas DataFrame of counts, or None if any column is not a string.
        """
        if not self._is_string(col1):
            print(f"Column '{col1}' is not a string column.")
            return None

        if col2 is not None:
            if not self._is_string(col2):
                print(f"Column '{col2}' is not a string column.")
                return None
            group_cols = [col1, col2]
        else:
            group_cols = [col1]

        result = (
            self.df
            .groupBy(*group_cols)
            .count()
            .orderBy(*group_cols)
            .toPandas()
        )
        return result

Writing SparkDataCheck.py


---
## Validating the Classe Created with dummy data

Before using the class in a real dataset I am testing how it would work with dummy controled data

In [2]:
!pip install pyspark -q

In [3]:
print("Step 1: importing...")
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd, io, contextlib, importlib
from IPython.display import display, Markdown

display(Markdown("### Step 1: OK"))

print("Step 2: starting Spark...")
spark = SparkSession.builder.appName("TestSDC").getOrCreate()
# spark.sparkContext.setLogLevel("ERROR") # Removed to show all outputs
display(Markdown(f"### Step 2: OK — Spark {spark.version}"))

print("Step 3: importing class...")
import SparkDataCheck as _m; importlib.reload(_m)
from SparkDataCheck import SparkDataCheck
display(Markdown("### Step 3: OK"))

print("Step 4: building test data...")
sdf = spark.createDataFrame(
    [(1, 10.0, "A"), (2, 25.0, "B"), (3, None, "C"), (4, 50.0, None), (5, 100.0, "A")],
    "val INT, score DOUBLE, label STRING"
)
print("Original DataFrame (sdf):")
display(sdf.toPandas()) # Convert to Pandas and display
display(Markdown("### Step 4: OK"))

print("Step 5: testing check_numeric_bounds...")
c = SparkDataCheck(sdf).check_numeric_bounds("score", lower=0, upper=50)
print("Result of check_numeric_bounds:")
display(c.df.select("score", "score_in_bounds").toPandas()) # Convert to Pandas and display
display(Markdown("### Step 5: OK"))

print("Step 6: testing check_missing...")
c = SparkDataCheck(sdf).check_missing("score")
print("Result of check_missing:")
display(c.df.select("score", "score_is_missing").toPandas()) # Convert to Pandas and display
display(Markdown("### Step 6: OK"))

print("Step 7: testing numeric_summary...")
res = SparkDataCheck(sdf).numeric_summary(col_name="score")
display(res)
display(Markdown("### Step 7: OK"))

display(Markdown("\n### All steps done!"))

Step 1: importing...


### Step 1: OK

Step 2: starting Spark...


### Step 2: OK — Spark 4.0.2

Step 3: importing class...


...

### Step 6: OK

Step 7: testing numeric_summary...


,score_min,score_max
0,10.0,100.0


### Step 7: OK


### All steps done!

---
## Using `SparkDataCheck` on the Air Quality Dataset

After simple testing now I am put the class to work in a real dataset. The dataset we'll use tracks daily air quality measurements (ozone, solar radiation, wind, and temperature) recorded in New York from May to September 1973.

We start by loading it directly from a URL using our `from_csv` class method, which reads the file into a Spark DataFrame and wraps it in a `SparkDataCheck` object — ready to use immediately.

In [5]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from IPython.display import display, Markdown
import pandas as pd, importlib, io, contextlib

spark = SparkSession.builder.appName("Project2").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 4.0.2


In [6]:
import SparkDataCheck as _m
importlib.reload(_m)
from SparkDataCheck import SparkDataCheck
print("SparkDataCheck ready")

SparkDataCheck ready


In [8]:
import urllib.request
urllib.request.urlretrieve("https://www4.stat.ncsu.edu/online/datasets/air.csv", "air.csv")

air = SparkDataCheck.from_csv(spark, "air.csv")

print(f"Rows: {air.df.count()}  |  Columns: {len(air.df.columns)}")
air.df.printSchema()
display(air.df.toPandas().head(10))

Rows: 9357  |  Columns: 16
root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



,_c0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,2026-03-21 18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,2026-03-21 19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,2026-03-21 20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,2026-03-21 21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,2026-03-21 22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888
5,5,3/10/2004,2026-03-21 23:00:00,1.2,1197,38,4.7,750,89,1337,96,1393,949,11.2,59.2,0.7848
6,6,3/11/2004,2026-03-21 00:00:00,1.2,1185,31,3.6,690,62,1462,77,1333,733,11.3,56.8,0.7603
7,7,3/11/2004,2026-03-21 01:00:00,1.0,1136,31,3.3,672,62,1453,76,1333,730,10.7,60.0,0.7702
8,8,3/11/2004,2026-03-21 02:00:00,0.9,1094,24,2.3,609,45,1579,60,1276,620,10.7,59.7,0.7648
9,9,3/11/2004,2026-03-21 03:00:00,0.6,1010,19,1.7,561,-200,1705,-200,1235,501,10.3,60.2,0.7517


---
### `check_numeric_bounds`

This method checks whether numeric values fall within a user-defined range and appends a boolean column to the DataFrame. We test both bounds together, each bound individually, a non-numeric column (to trigger the warning), and chaining.